In [ ]:
import pandas as pd
import json
import os
import requests
import csv
import kagglehub
import glob

import re
import html
    
def standardize_label(label, dataset_name):
    """Maps dataset-specific labels strictly to a 3-class taxonomy."""
    label = str(label).lower().strip()

    if dataset_name == "PUBHEALTH":
        if label == "true": return "SUPPORTED"
        if label == "false": return "REFUTED"
        if label == "unproven": return "NOT_ENOUGH_INFO"
        return None 

    elif dataset_name == "FEVER":
        if label == "supports": return "SUPPORTED"
        if label == "refutes": return "REFUTED"
        if label == "not enough info": return "NOT_ENOUGH_INFO"
        return None 

    elif dataset_name == "MultiFC":
        # Check partials first
        if any(x in label for x in ["mostly true", "mostly accurate"]):
            return "SUPPORTED"
        if any(x in label for x in ["mostly false", "barely true"]):
            return "REFUTED"
        if any(x in label for x in ["half-true", "half true", "mixture", "mixed"]):
            return None
        if any(x in label for x in ["unproven", "no evidence", "inconclusive"]):
            return "NOT_ENOUGH_INFO"
           
        # Check pure truths/falsehoods
        if any(x in label for x in ["true", "accurate", "supported", "correct"]):
            return "SUPPORTED"
        if any(x in label for x in ["false", "fake", "fiction", "incorrect", "pants on fire"]):
            return "REFUTED"
            
        return None
        
    return None

# 1. Update loaders to return (df, raw_total_count)
def load_pubhealth(file_path):
    df = pd.read_csv(file_path, sep='\t', on_bad_lines='skip')
    raw_count = len(df)
    
    standardized_data = []
    for idx, row in df.iterrows():
        mapped_label = standardize_label(row['label'], "PUBHEALTH")
        if mapped_label:
            standardized_data.append({
                "id": f"pubhealth_{idx}",
                "claim": str(row['claim']) if pd.notnull(row['claim']) else "",
                "evidence": str(row['explanation']) if pd.notnull(row['explanation']) else "",
                "label": mapped_label,
                "original_label": str(row['label']),
                "dataset": "PUBHEALTH"
            })
    return pd.DataFrame(standardized_data), raw_count

def load_multifc(file_path):
    columns = ["claimID", "claim", "label", "claimURL", "reason", "category", "speaker", "checker", "tags", "articleTitle", "publishDate", "claimDate", "entities"]
    df = pd.read_csv(file_path, sep='\t', header=None, names=columns)
    raw_count = len(df)
    
    standardized_data = []
    for _, row in df.iterrows():
        mapped_label = standardize_label(row['label'], "MultiFC")
        if mapped_label:
            standardized_data.append({
                "id": f"multifc_{row['claimID']}",
                "claim": str(row['claim']),
                "evidence": str(row['reason']) if pd.notnull(row['reason']) else "",
                "label": mapped_label,
                "original_label": str(row['label']),
                "dataset": "MultiFC"
            })
    return pd.DataFrame(standardized_data), raw_count

def load_fever(file_path, wiki_db):
    standardized_data = []
    raw_count = 0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line)
                raw_count += 1
                
                original_label = str(item.get('label', ''))
                mapped_label = standardize_label(original_label, "FEVER")
                
                if mapped_label:
                    evidence_list = item.get('evidence', [])
                    evidence_texts = []
                    
                    for evidence_group in evidence_list:
                        for evidence in evidence_group:
                            if isinstance(evidence, list) and len(evidence) >= 4:
                                wiki_url = evidence[2]
                                sentence_id = evidence[3]
                                
                                if wiki_url and wiki_url in wiki_db:
                                    try:
                                        sent_id_int = int(sentence_id)
                                        sent_text = wiki_db[wiki_url].get(sent_id_int, "")
                                        if sent_text:
                                            sent_text = sent_text.replace("-LRB-", "(").replace("-RRB-", ")")
                                            evidence_texts.append(sent_text)
                                    except (ValueError, TypeError):
                                        pass
                    
                    final_evidence = " ".join(evidence_texts)

                    standardized_data.append({
                        "id": f"fever_{item.get('id', len(standardized_data))}",
                        "claim": item.get("claim", ""),
                        "evidence": final_evidence,
                        "label": mapped_label,
                        "original_label": original_label,
                        "dataset": "FEVER"
                    })
            except json.JSONDecodeError:
                continue
                
    return pd.DataFrame(standardized_data), raw_count

# 2. Track raw counts during loading
dataframes = []
raw_counts = {}

print("Processing FEVER...")
wiki_mapping = load_fever_wiki("data/wiki-pages")
df_fever, raw_counts["FEVER"] = load_fever("data/fever_train.jsonl", wiki_mapping)
dataframes.append(df_fever)

print("Processing PUBHEALTH...")
df_pub, raw_counts["PUBHEALTH"] = load_pubhealth(r"data/pubhealth_train.tsv")
dataframes.append(df_pub)

print("Processing MultiFC...")
df_mfc, raw_counts["MultiFC"] = load_multifc("data/multifc_train.tsv")
dataframes.append(df_mfc)

if dataframes:
    # ---- Balancing Step ----
    # 1. Determine target number for SUPPORTED (average of REFUTED and NOT_ENOUGH_INFO)
    total_refuted = sum(len(df[df['label'] == 'REFUTED']) for df in dataframes)
    total_nei = sum(len(df[df['label'] == 'NOT_ENOUGH_INFO']) for df in dataframes)
    target_supported_total = int((total_refuted + total_nei) / 2)
    
    # 2. Calculate how many total SUPPORTED claims need to be removed
    total_supported_current = sum(len(df[df['label'] == 'SUPPORTED']) for df in dataframes)
    total_to_remove = max(0, total_supported_current - target_supported_total)
    
    # 3. Distribute the removals equally across all datasets adaptively
    counts = [len(df[df['label'] == 'SUPPORTED']) for df in dataframes]
    removed_counts = [0] * len(dataframes)
    remaining_to_remove = total_to_remove
    
    while remaining_to_remove > 0:
        active_indices = [i for i in range(len(dataframes)) if counts[i] > removed_counts[i]]
        if not active_indices:
            break
        
        # Determine how many to remove equally per active dataset in this step
        amt = min(remaining_to_remove // len(active_indices), min(counts[i] - removed_counts[i] for i in active_indices))
        if amt == 0:
            amt = 1  # Handle remaining odd numbers
            
        for i in active_indices:
            if remaining_to_remove >= amt:
                removed_counts[i] += amt
                remaining_to_remove -= amt
            else:
                removed_counts[i] += remaining_to_remove
                remaining_to_remove = 0
                break

    # 4. Filter and sample each dataset down
    balanced_dataframes = []
    dataset_names = ["FEVER", "PUBHEALTH", "MultiFC"]
    
    print("\n--- Balancing Datasets (Per-Dataset SUPPORTED Class Downsampling) ---")
    for idx, df in enumerate(dataframes):
        if df.empty:
            balanced_dataframes.append(df)
            continue
            
        supported_df = df[df['label'] == 'SUPPORTED']
        refuted_count = len(df[df['label'] == 'REFUTED'])
        nei_count = len(df[df['label'] == 'NOT_ENOUGH_INFO'])
        
        # Target for this specific dataset
        target_supported = int((refuted_count + nei_count) / 2)
        
        if len(supported_df) > target_supported:
            num_to_remove = len(supported_df) - target_supported
            print(f"{dataset_names[idx]}: Removing {num_to_remove} out of {len(supported_df)} 'SUPPORTED' claims.")
            supported_balanced = supported_df.sample(n=target_supported, random_state=42)
        else:
            print(f"{dataset_names[idx]}: No downsampling needed (already below target).")
            supported_balanced = supported_df
            
        other_df = df[df['label'] != 'SUPPORTED']
        df_balanced = pd.concat([supported_balanced, other_df], ignore_index=True)
        balanced_dataframes.append(df_balanced)

    unified_df = pd.concat(balanced_dataframes, ignore_index=True)
    
    if 'evidence' in unified_df.columns:
        unified_df['evidence'] = unified_df['evidence'].fillna("None")
    
    output_path = "dataset.jsonl"
    unified_df.to_json(output_path, orient="records", lines=True)
    print(f"\nSuccessfully combined datasets into 3 classes. Saved to {output_path}")
    print(f"Total records: {len(unified_df)}")
    print("\nLabel Distribution:")
    print(unified_df['label'].value_counts())
    print("\nDataset Distribution:")
    print(unified_df['dataset'].value_counts())
    print("\nPer-Dataset Label Distribution:")
    print(pd.crosstab(unified_df['dataset'], unified_df['label'], margins=True))

    print("\nDiscarded Samples Breakdown:")
    raw_df_map = dict(zip(["FEVER", "PUBHEALTH", "MultiFC"], dataframes))
    
    summary = []
    for name in ["FEVER", "PUBHEALTH", "MultiFC"]:
        initial = raw_counts[name]
        after_label_filter = len(raw_df_map[name])
        final_kept = len(unified_df[unified_df['dataset'] == name])
        
        summary.append({
            "Dataset": name,
            "Initial Raw": initial,
            "Dropped (Unmapped Label)": initial - after_label_filter,
            "Dropped (Downsampling)": after_label_filter - final_kept,
            "Total Discarded": initial - final_kept,
            "Final Kept": final_kept
        })
        
    print(pd.DataFrame(summary).to_string(index=False))

Processing FEVER...
Loading FEVER Wikipedia corpus into memory...
Processing PUBHEALTH...
Processing MultiFC...

--- Balancing Datasets (Per-Dataset SUPPORTED Class Downsampling) ---
FEVER: Removing 47328 out of 80035 'SUPPORTED' claims.
PUBHEALTH: Removing 3432 out of 5078 'SUPPORTED' claims.
MultiFC: No downsampling needed (already below target).

Successfully combined datasets into 3 classes. Saved to unified_fact_checking_data_3class.jsonl
Total records: 120662

Label Distribution:
label
REFUTED            43937
SUPPORTED          40170
NOT_ENOUGH_INFO    36555
Name: count, dtype: int64

Dataset Distribution:
dataset
FEVER        98121
MultiFC      17603
PUBHEALTH     4938
Name: count, dtype: int64

Per-Dataset Label Distribution:
label      NOT_ENOUGH_INFO  REFUTED  SUPPORTED     All
dataset                                               
FEVER                35639    29775      32707   98121
MultiFC                625    11161       5817   17603
PUBHEALTH              291     3001